# NB_lab_test_trends
Builds `<catalog>.gold.lab_test_trends` per the STTM (`sttm_silver_to_gold.xlsx`, sheet *lab_test_trends*).

* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `<catalog>.silver.lab_results` grouped by `test_name`, `year(test_date)`, `month(test_date)`
* **Joins (LEFT, optional):** calendar on `date_key = yyyyMMdd(first day of month)`
* **Window:** `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year`, `month`
* **Load:** FULL - `CREATE OR REPLACE TABLE ... AS SELECT`

Note: the calendar table exists in silver as `calender` (STTM spells it `calendar`).

In [0]:
dbutils.widgets.text("catalog", "adb_caren")
catalog = dbutils.widgets.get("catalog").strip()

source_tables = ["lab_results", "calender"]
missing = [t for t in source_tables if not spark.catalog.tableExists(f"{catalog}.silver.{t}")]
if missing:
    raise ValueError(f"Missing source tables in {catalog}.silver: {missing}")

spark.sql("CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.gold')", args={"catalog": catalog})

In [0]:
lab_test_trends_sql = """
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.lab_test_trends') AS
WITH monthly AS (
  SELECT test_name,
         year(test_date)                      AS year,
         month(test_date)                     AS month,
         COUNT(*)                             AS total_tests,
         COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
         COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM IDENTIFIER(:catalog || '.silver.lab_results')
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT m.*,
         CAST(ROUND(abnormal_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
         CAST(ROUND(critical_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT r.*,
         LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name,
  CAST(w.year AS INT)                 AS year,
  CAST(w.month AS INT)                AS month,
  c.month_name,
  c.quarter_name,
  CAST(w.total_tests AS INT)          AS total_tests,
  CAST(w.abnormal_test_count AS INT)  AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)  AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  w.prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN IDENTIFIER(:catalog || '.silver.calender') c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT)
"""

spark.sql(lab_test_trends_sql, args={"catalog": catalog})
print(f"Rebuilt {catalog}.gold.lab_test_trends")

In [0]:
gold_df = spark.table(f"{catalog}.gold.lab_test_trends")
gold_count = gold_df.count()
distinct_keys = gold_df.select("test_name", "year", "month").distinct().count()
gold_tests = gold_df.agg({"total_tests": "sum"}).collect()[0][0]
source_tests = spark.table(f"{catalog}.silver.lab_results").count()

print(f"gold rows: {gold_count} | tests in gold: {gold_tests} | tests in silver: {source_tests}")
assert distinct_keys == gold_count, "Duplicate (test_name, year, month) found - grain violated"
assert gold_tests == source_tests, "Total tests do not reconcile with silver.lab_results"

display(gold_df.orderBy("test_name", "year", "month"))

In [0]:
# Final summary
target_table = f"{catalog}.gold.lab_test_trends"
load_type_normalized = "FULL"

result_df = spark.read.table(target_table)
result_count = result_df.count()
print(f"\n{'='*60}")
print(f"✅ {load_type_normalized} load completed for {target_table}")
print(f"📊 Total rows in target table after load: {result_count}")
print(f"{'='*60}")

dbutils.notebook.exit(str(result_count))